In [ ]:
# -*- coding: utf-8 -*-
import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import solve
from scipy.signal import find_peaks, peak_widths
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 50, 70}

# RF para temperatura (curva -> T)
RF_TEMP_PARAMS = dict(
    n_estimators=800, max_depth=24,
    min_samples_leaf=2, min_samples_split=4,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=7
)

# RFs multi-output (um por alvo: shift, gain, delta)
RF_SHIFT_PARAMS = dict(
    n_estimators=900, max_depth=22,
    min_samples_leaf=2, min_samples_split=6,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=101
)
RF_GAIN_PARAMS  = dict(
    n_estimators=900, max_depth=22,
    min_samples_leaf=2, min_samples_split=6,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=102
)
RF_DELTA_PARAMS = dict(
    n_estimators=900, max_depth=22,
    min_samples_leaf=2, min_samples_split=6,
    max_features="sqrt", bootstrap=True,
    n_jobs=-1, random_state=103
)

# ---------- Picos / janelas ----------
REF_PEAK_PROM_FRAC   = 0.12   # fração da amplitude para detectar picos na ref
REF_MIN_PEAK_DIST_HZ = 250.0  # distância mínima entre picos
LABEL_SHIFT_WIN_HZ   = 800.0  # busca de shift no treino (±)  **reduzi**
LOCAL_BAND_HZ        = 900.0  # faixa para LS de g/δ ao redor do pico
NORM_EPS             = 1e-12

# ---------- caps e suavização ----------
CAP_SIGMA_PER_FREQ   = 3.0    # limite por desvio-padrão por frequência (aplicação)
CAP_FRAC_AMP         = 0.35   # limite por fração da amplitude da amostra
SMOOTH_WIN           = 5      # janela ímpar p/ suavização final

# ===================== HELPERS =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    r=win//2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s/float(win)

# ===== métricas (Dias) =====
def rmsd(y_ref, y):   return float(np.sqrt(np.mean((y_ref - y)**2)))
def ccdm(y_ref, y):
    y1, y2 = y_ref - y_ref.mean(), y - y.mean()
    den = (np.linalg.norm(y1)*np.linalg.norm(y2))+1e-12
    rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
    return 1.0 - rho
def corr_per_sample(Y, Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(float(np.clip(num/den,-1,1)))
    return np.array(out)
def sam_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=float(np.clip(np.dot(y,yh)/den,-1,1))
        out.append(float(np.degrees(np.arccos(cosang))))
    return np.array(out)
def nrmse_per_sample(Y,Yhat):
    out=[]
    for y,yh in zip(Y,Yhat):
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(float(rmse/(rng+1e-12)))
    return np.array(out)
def eval_all_metrics(Y_true, Y_pred):
    y1, y2 = Y_true.reshape(-1), Y_pred.reshape(-1)
    return dict(
        R2      = r2_score(y1, y2),
        RMSE    = float(np.sqrt(mean_squared_error(y1, y2))),
        MAE     = float(mean_absolute_error(y1, y2)),
        Corr    = float(corr_per_sample(Y_true, Y_pred).mean()),
        SAM_deg = float(sam_per_sample(Y_true, Y_pred).mean()),
        NRMSE   = float(nrmse_per_sample(Y_true, Y_pred).mean()),
        RMSD    = float(np.mean([rmsd(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
        CCDM    = float(np.mean([ccdm(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
    )
def print_metrics_block(title, m):
    print(f"\n== {title} ==")
    print(" | ".join([f"{k}={m[k]:.4f}" for k in ["R2","RMSE","MAE","Corr","SAM_deg","NRMSE","RMSD","CCDM"]]))

# ===== picos =====
def detect_peaks(y, fhz, prom_frac, min_dist_hz):
    amp = y.max() - y.min()
    prom = max(1e-6, prom_frac * amp)
    df = float(np.median(np.diff(fhz)))
    dist_pts = max(1, int(round(min_dist_hz/df)))
    peaks, props = find_peaks(y, prominence=prom, distance=dist_pts)
    return peaks, props

def band_indices_around(idx_center, fhz, band_hz):
    df = float(np.median(np.diff(fhz)))
    r = max(2, int(round(band_hz/df)))  # >=2 para garantir >=5 pontos no total
    j0 = max(0, idx_center - r)
    j1 = min(len(fhz), idx_center + r + 1)
    return j0, j1

# ===== rótulos no TREINO (shift/gain/delta por âncora) =====
def ls_gain_delta(y_tgt, x_src):
    one = np.ones_like(x_src)
    a11 = np.dot(x_src, x_src) + 1e-12
    a12 = np.dot(x_src, one)
    a22 = len(x_src)
    b1  = np.dot(x_src, y_tgt)
    b2  = np.dot(one,  y_tgt)
    A = np.array([[a11, a12],[a12, a22]], float)
    b = np.array([b1,  b2],  float)
    g, d = solve(A, b)
    return float(g), float(d)

def best_shift_gain_delta_for_anchor_FAST(x, y_ref, fhz, idx_ref_k,
                                          search_win_hz, band_hz):
    """
    Versão rápida: shifts em múltiplos INTEIROS de Δf.
    Usa SLICES alinhados (sem interpolação) e calcula LS em cada shift.
    """
    m = len(fhz); df = float(np.median(np.diff(fhz)))
    s_pts = max(1, int(round(search_win_hz/df)))

    # janela em torno do pico de referência
    j0, j1 = band_indices_around(idx_ref_k, fhz, band_hz)  # [j0:j1)
    best = (np.inf, 0, 1.0, 0.0)  # mse, s_idx, g, d

    for s in range(-s_pts, s_pts+1):
        # alinhar janelas (zona comum após o shift)
        ia0 = max(0, j0 - s); ia1 = min(m, j1 - s)   # no x deslocado
        r0  = ia0 + s; r1  = ia1 + s                 # no y_ref
        if ia1-ia0 < 5:   # janela curta -> ignora
            continue
        xb = x[ia0:ia1]
        yb = y_ref[r0:r1]
        g, d = ls_gain_delta(yb, xb)
        yhat = g*xb + d
        mse = np.mean((yhat - yb)**2)
        if mse < best[0]:
            best = (mse, s, g, d)

    _, s_idx, g, d = best
    return s_idx*df, g, d  # em Hz

# ===== features =====
def global_feats(Xrow):
    mu = Xrow.mean(); sd = Xrow.std(); amp = Xrow.max() - Xrow.min()
    z = (Xrow - mu)/(sd + NORM_EPS)
    s3 = float(np.mean(z**3)); s4 = float(np.mean(z**4))
    return np.array([mu, sd, amp, s3, s4], float)

def safe_slope_curv(seg, fseg):
    # precisa de >=3 pontos e f crescente
    if len(seg) < 3: return 0.0, 0.0
    df = np.diff(fseg)
    if not np.all(df > 0):  # proteção
        return 0.0, 0.0
    # derivadas médias simples (numericamente estáveis)
    s = np.gradient(seg, fseg, edge_order=1)
    c = np.gradient(s,   fseg, edge_order=1)
    return float(np.mean(s)), float(np.mean(c))

def local_feats(Xrow, fhz, idx_center, band_hz):
    j0, j1 = band_indices_around(idx_center, fhz, band_hz)
    seg = Xrow[j0:j1]; fseg = fhz[j0:j1]
    if len(seg) < 5:  # janela mínima
        return np.zeros(7, float)
    mu = seg.mean(); sd = seg.std(); amp = seg.max() - seg.min()
    slope, curv = safe_slope_curv(seg, fseg)
    pk, props = find_peaks(seg, prominence=(0.05*amp))
    if len(pk)>0:
        prom = float(np.max(props["prominences"]))
        w = float(np.mean(peak_widths(seg, pk, rel_height=0.5)[0]))
    else:
        prom, w = 0.0, 0.0
    return np.array([mu, sd, amp, slope, curv, prom, w], float)

def build_training_tables(X, T, fhz, y_ref, ref_peaks_idx):
    """
    Retorna:
      X_tab  -> (n, D) (média das features de todas âncoras; D inclui f_k e k_norm)
      y_shift, y_gain, y_delta -> (n, K)
      sd_f (p/ cap por frequência na aplicação)
    """
    n, m = X.shape
    K = len(ref_peaks_idx)
    fmin, fmax = fhz.min(), fhz.max()

    # stats p/ cap na aplicação
    resid = (y_ref[None,:] - X)
    sd_f = resid.std(axis=0) + NORM_EPS

    X_rows = []
    y_shift = np.zeros((n, K), float)
    y_gain  = np.zeros((n, K), float)
    y_delta = np.zeros((n, K), float)

    for i in range(n):
        row = X[i]; dT = float(T[i] - REF_TEMP)
        gfeat = global_feats(row)
        Xi=[]
        for kk, ir in enumerate(ref_peaks_idx):
            # rótulos rápidos (apenas treino)
            s_hz, g, d = best_shift_gain_delta_for_anchor_FAST(
                row, y_ref, fhz, ir, LABEL_SHIFT_WIN_HZ, LOCAL_BAND_HZ
            )
            y_shift[i, kk] = s_hz
            y_gain [i, kk] = g
            y_delta[i, kk] = d

            # features
            f_k = fhz[ir]
            locf = local_feats(row, fhz, ir, LOCAL_BAND_HZ)
            Xi.append(np.concatenate([
                [dT, dT**2, dT**3], gfeat,
                [(f_k - fmin)/(fmax-fmin+NORM_EPS), kk/(K-1 if K>1 else 1.0)],
                locf
            ]))
        X_rows.append(np.vstack(Xi))  # (K, D)

    X_rows = np.stack(X_rows, axis=0)  # (n, K, D)
    X_tab  = X_rows.mean(axis=1)       # (n, D) — agrego, mas D traz f_k, k_norm
    return X_tab, y_shift, y_gain, y_delta, sd_f

def build_test_features(X, T_pred, fhz, y_ref, ref_peaks_idx):
    n, m = X.shape
    K = len(ref_peaks_idx)
    fmin, fmax = fhz.min(), fhz.max()
    X_rows=[]
    for i in range(n):
        row = X[i]; dT = float(T_pred[i] - REF_TEMP)
        gfeat = global_feats(row)
        Xi=[]
        for kk, ir in enumerate(ref_peaks_idx):
            f_k = fhz[ir]
            locf = local_feats(row, fhz, ir, LOCAL_BAND_HZ)
            Xi.append(np.concatenate([
                [dT, dT**2, dT**3], gfeat,
                [(f_k - fmin)/(fmax-fmin+NORM_EPS), kk/(K-1 if K>1 else 1.0)],
                locf
            ]))
        X_rows.append(np.vstack(Xi))
    X_rows = np.stack(X_rows, axis=0)
    X_tab  = X_rows.mean(axis=1)
    return X_tab

def enforce_monotonic_shifts(f_ref, shifts_hz):
    f_new = f_ref + shifts_hz
    for k in range(1, len(f_new)):
        if f_new[k] <= f_new[k-1] + 1e-6:
            f_new[k] = f_new[k-1] + 1e-6
    return f_new

def apply_piecewise_affine(x, fhz, f_ref_peaks, f_new_peaks, gain_k, delta_k):
    # bordas como âncoras
    f_src_anc = np.concatenate([[fhz[0]], f_ref_peaks, [fhz[-1]]])
    f_dst_anc = np.concatenate([[fhz[0] + (f_new_peaks[0]-f_ref_peaks[0])],
                                f_new_peaks,
                                [fhz[-1] + (f_new_peaks[-1]-f_ref_peaks[-1])]])
    f_mapped = np.interp(fhz, f_src_anc, f_dst_anc)
    x_warp = np.interp(fhz, f_mapped, x, left=x[0], right=x[-1])

    # ganho/offset por frequência (interpolados entre âncoras)
    f1 = np.concatenate([[fhz[0]], f_new_peaks, [fhz[-1]]])
    g1 = np.concatenate([[gain_k[0]], gain_k, [gain_k[-1]]])
    d1 = np.concatenate([[delta_k[0]], delta_k, [delta_k[-1]]])
    g_f = np.interp(fhz, f1, g1)
    d_f = np.interp(fhz, f1, d1)
    return g_f*x_warp + d_f

# ===================== LOAD & PREP =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz); common_cols = [common_cols[i] for i in order]; fhz = fhz[order]
fkHz = fhz/1e3

X_tr_full = base_tr[common_cols].to_numpy(float)
X_te_full = base_te[common_cols].to_numpy(float)
T_tr_full = base_tr["temp_c"].to_numpy(float)
T_te_full = base_te["temp_c"].to_numpy(float)

# referência @20 °C
pool_20=[]
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
assert len(pool_20)>0, "Não há curva real @20°C!"
y_ref = np.median(np.vstack(pool_20), axis=0)

# RF-Temp (ΔT como feature)
X_all = np.vstack([X_tr_full, X_te_full])
T_all = np.concatenate([T_tr_full, T_te_full])
rf_temp = RandomForestRegressor(**RF_TEMP_PARAMS).fit(X_all, T_all)
print("\n== RF-Temp ==")
print(f"R²(all)={r2_score(T_all, rf_temp.predict(X_all)):.3f} | RMSE(all)={np.sqrt(mean_squared_error(T_all, rf_temp.predict(X_all))):.3f}")

# Conjuntos fixos
tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()
X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# Âncoras na referência
ref_peaks_idx, ref_props = detect_peaks(y_ref, fhz, REF_PEAK_PROM_FRAC, REF_MIN_PEAK_DIST_HZ)
K = len(ref_peaks_idx)
assert K>=2, "Poucas âncoras detectadas na referência."

# ===================== DATASET DE TREINO (rótulos + features) =====================
t0=time.time()
Xtr_tab, y_shift, y_gain, y_delta, sd_f = build_training_tables(
    X_tr, T_tr, fhz, y_ref, ref_peaks_idx
)
print(f"[INFO] dataset treino montado em {time.time()-t0:.1f}s | n={Xtr_tab.shape[0]} | K={K}")

# ===================== TREINO RFs (multi-output) =====================
rf_shift = RandomForestRegressor(**RF_SHIFT_PARAMS).fit(Xtr_tab, y_shift)
rf_gain  = RandomForestRegressor(**RF_GAIN_PARAMS ).fit(Xtr_tab, y_gain )
rf_delta = RandomForestRegressor(**RF_DELTA_PARAMS).fit(Xtr_tab, y_delta)

# ===================== PREDIÇÃO (100% RF) =====================
T_te_pred = rf_temp.predict(X_te)
Xte_tab = build_test_features(X_te, T_te_pred, fhz, y_ref, ref_peaks_idx)

shift_hat = rf_shift.predict(Xte_tab)   # (n_te, K) em Hz
gain_hat  = rf_gain.predict(Xte_tab)    # (n_te, K)
delta_hat = rf_delta.predict(Xte_tab)   # (n_te, K)

Y_te_hat = np.zeros_like(X_te)
f_ref_peaks = fhz[ref_peaks_idx]

for i in range(X_te.shape[0]):
    x = X_te[i]
    amp_i = x.max() - x.min()
    cap_vec = np.minimum(CAP_SIGMA_PER_FREQ*sd_f, CAP_FRAC_AMP*amp_i)

    f_new_peaks = enforce_monotonic_shifts(f_ref_peaks, shift_hat[i])
    y_est = apply_piecewise_affine(x, fhz, f_ref_peaks, f_new_peaks,
                                   gain_hat[i], delta_hat[i])

    # segurança + suavização leve
    y_est = np.clip(y_est, x - cap_vec, x + cap_vec)
    y_est = moving_average(y_est, SMOOTH_WIN)
    Y_te_hat[i] = y_est

# ===================== MÉTRICAS =====================
Y_ref_te = np.tile(y_ref, (X_te.shape[0],1))
def metrics_block(title, Yt, Yp):
    m = eval_all_metrics(Yt, Yp); print_metrics_block(title, m); return m

print("\n### MÉTRICAS — RF Âncorado (100% RF na predição) ###")
m_vs_ref  = metrics_block("RF-Anchor vs REF",  Y_ref_te, Y_te_hat)
m_vs_orig = metrics_block("RF-Anchor vs ORIG", X_te,     Y_te_hat)

print("\n### Checagem RF-Temp (curvas finais) ###")
T_hat = rf_temp.predict(Y_te_hat)
print(f"média={float(np.mean(T_hat)):.2f}°C | desvio={float(np.std(T_hat)):.2f}°C | MAE vs {REF_TEMP}°C={float(np.mean(np.abs(T_hat-REF_TEMP))):.2f}°C")

# ===================== PLOT =====================
def _prep_plot():
    plt.rcParams.update({
        "figure.figsize": (8.6, 4.8),
        "axes.grid": True, "grid.alpha": 0.25,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelsize": 12, "axes.titlesize": 13,
        "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 10, "lines.linewidth": 1.8,
    })

def plot_model(i=0, save=False, prefix="rf_anchor_only"):
    _prep_plot()
    fhz_khz = fkHz
    T_real = float(te_restr.iloc[i]["temp_c"])
    T_prev = float(T_te_pred[i])
    fig, ax = plt.subplots()
    ax.plot(fhz_khz, X_te[i],    label=f"Original @ {T_real:.0f} °C",  color="#1f77b4")
    ax.plot(fhz_khz, y_ref,      label=f"Referência @ {REF_TEMP} °C", color="#ff7f0e")
    ax.plot(fhz_khz, Y_te_hat[i],label="RF (âncoras por pico)",       color="#2ca02c")
    ax.set_title(f"Amostra {i}  |  T real = {T_real:.0f} °C  |  T prev = {T_prev:.1f} °C")
    ax.set_xlabel("Frequência (kHz)"); ax.set_ylabel("Re{Z}")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    if save: fig.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

# Exemplo de plot
plot_model(i=0, save=False)
